## Transformers (english to telugu)

In [16]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.layers import (
    TextVectorization,
    Embedding,
    Dense,
    LayerNormalization,
    MultiHeadAttention
)
from tensorflow.keras import Model

# English to Telugu Dataset
data = [
    ("i am a student", "నేను ఒక విద్యార్థిని"),
    ("how are you", "మీరు ఎలా ఉన్నారు"),
    ("i love machine learning", "నేను మెషిన్ లెర్నింగ్‌ను ప్రేమిస్తున్నాను"),
    ("good morning", "శుభోదయం"),
    ("thank you", "ధన్యవాదాలు"),
    ("see you later", "తర్వాత కలుద్దాం"),
    ("what is your name", "మీ పేరు ఏమిటి"),
    ("where are you going", "మీరు ఎక్కడికి వెళ్తున్నారు"),
    ("i like coffee", "నాకు కాఫీ అంటే ఇష్టం"),
    ("welcome", "స్వాగతం")
]

english_sentences = [x[0] for x in data]

# CRITICAL FIX: Added spaces around 'start' and 'end' so they tokenize cleanly
telugu_sentences = ["start " + x[1] + " end" for x in data]

In [ ]:
vocab_size = 1000 
sequence_length = 20 

source_vectorization = TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length
)

target_vectorization = TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length
)

source_vectorization.adapt(english_sentences)
target_vectorization.adapt(telugu_sentences)

encoder_inputs = source_vectorization(english_sentences)
target_tokens = target_vectorization(telugu_sentences)

decoder_inputs = target_tokens[:, :-1]
decoder_outputs = target_tokens[:, 1:]

In [18]:
class PositionalEncoding(tf.keras.layers.Layer):
    def __init__(self, sequence_length, vocab_size, embed_dim):
        super().__init__()
        self.token_embedding = Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.position_embedding = Embedding(input_dim=sequence_length, output_dim=embed_dim)
        self.sequence_length = sequence_length

    def call(self, inputs):
        length = tf.shape(inputs)[-1]
        positions = tf.range(start=0, limit=length, delta=1)
        embedded_tokens = self.token_embedding(inputs)
        embedded_positions = self.position_embedding(positions)
        return embedded_tokens + embedded_positions

In [19]:
class TransformerEncoderLayer(tf.keras.layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads):
        super().__init__()
        self.attention = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.dense_proj = tf.keras.Sequential([
            Dense(dense_dim, activation="relu"),
            Dense(embed_dim)
        ])
        self.layernorm1 = LayerNormalization()
        self.layernorm2 = LayerNormalization()

    def call(self, inputs):
        attention_output = self.attention(inputs, inputs)
        proj_input = self.layernorm1(inputs + attention_output)
        proj_output = self.dense_proj(proj_input)
        return self.layernorm2(proj_input + proj_output)

In [20]:
class TransformerDecoder(tf.keras.layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads):
        super().__init__()
        self.self_attention = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.cross_attention = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential([
            Dense(dense_dim, activation="relu"),
            Dense(embed_dim)
        ])
        self.layernorm1 = LayerNormalization()
        self.layernorm2 = LayerNormalization()
        self.layernorm3 = LayerNormalization()

    def call(self, inputs, encoder_outputs):
        attention_output = self.self_attention(inputs, inputs, inputs, use_causal_mask=True)
        out1 = self.layernorm1(inputs + attention_output)
        attention_output2 = self.cross_attention(out1, encoder_outputs)
        out2 = self.layernorm2(out1 + attention_output2)
        ffn_output = self.ffn(out2)
        return self.layernorm3(out2 + ffn_output)

In [21]:
embed_dim = 128
dense_dim = 512
num_heads = 4

encoder_input = tf.keras.Input(shape=(None,), dtype="int64")
x = PositionalEncoding(sequence_length, vocab_size, embed_dim)(encoder_input)
encoder_output = TransformerEncoderLayer(embed_dim, dense_dim, num_heads)(x)

decoder_input = tf.keras.Input(shape=(None,), dtype="int64")
x = PositionalEncoding(sequence_length, vocab_size, embed_dim)(decoder_input)
x = TransformerDecoder(embed_dim, dense_dim, num_heads)(x, encoder_output)
decoder_output = Dense(vocab_size, activation="softmax")(x)

transformer = Model([encoder_input, decoder_input], decoder_output)
transformer.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

In [27]:
transformer.fit(
    [encoder_inputs, decoder_inputs],
    decoder_outputs,
    batch_size=2,
    epochs=20, 
)

Epoch 1/20


5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 1.0000 - loss: 0.0048
Epoch 2/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 1.0000 - loss: 0.0046
Epoch 3/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 1.0000 - loss: 0.0045
Epoch 4/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 1.0000 - loss: 0.0043
Epoch 5/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 1.0000 - loss: 0.0042
Epoch 6/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 1.0000 - loss: 0.0041
Epoch 7/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 1.0000 - loss: 0.0040
Epoch 8/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 1.0000 - loss: 0.0038
Epoch 9/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 1.0000 - loss: 0.0037
Epoch 10/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 1.0000 - loss: 0.0036
Epoch 11/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 1.0000 - loss: 0.0035
Epoch 12/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 1.0000 - loss: 0.0034
Epoch 13/20


In [29]:
test_sentence = [" how are you "]
encoder_input_test = source_vectorization(test_sentence)

start_sentence = "start"
decoded_sentence = start_sentence

vocabulary = target_vectorization.get_vocabulary()
index_lookup = dict(zip(range(len(vocabulary)), vocabulary))

for i in range(10):
    tokenized_target = target_vectorization([decoded_sentence])
    
    predictions = transformer.predict(
        [encoder_input_test, tokenized_target],
        verbose=0
    )
    
    # Target index based on the actual lengths populated in our sentence array
    current_length = np.count_nonzero(tokenized_target[0])
    sampled_token_index = np.argmax(predictions[0, current_length - 1, :])
    sampled_token = index_lookup[sampled_token_index]
    
    if sampled_token == "" or sampled_token == "[UNK]":
        break
        
    decoded_sentence += " " + sampled_token
    
    if sampled_token == "end":
        break

# Clean out structural wrappers for final display
final_output = decoded_sentence.replace("start", "").replace("end", "").strip()

print("\n--- Corrected Translation Output ---")
print(f"Input:  {test_sentence[0]}")
print(f"Output: {final_output}")


--- Corrected Translation Output ---
Input:   how are you 
Output: మీరు ఎలా ఉన్నారు
